## UN COMTRADE 데이터 10만건 이상 다운로드 받기 
---


**1. 패키지 및 환경설정**

In [1]:
# 패키지 설정
import pandas as pd
import numpy as np
import os
import time
import json
import warnings
import requests
from io import BytesIO
from tqdm import tqdm
import time
import os

# 저장 폴더 생성 
os.makedirs('data/raw/comtrade', exist_ok=True)
os.makedirs('data/output', exist_ok=True)

print('import & 폴더 생성 완료')

import & 폴더 생성 완료


In [2]:
# SSL 검증 끄기 
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# API 키 설정 & URL 설정 
COMTRADE_API_KEY = "1d4e89d295bc4d97afa080099b5151bf"  # 본인 API 키를 넣어야 함 
BASE_URL = "https://comtradeapi.un.org/data/v1/get/C/A/HS"

print('SSL 검증 끔 & URL, API 설정 완료')

SSL 검증 끔 & URL, API 설정 완료


** 2. 국가 코드 가져오기 ** 
- 문제 : UNcomtrade API는 1회 10만건 이상 파싱 하지 못하게 함 
- 해결 : 국가코드를 청크별로 나눠서 데이터 다운로드 

In [5]:
#--- 국가 코드 가져오기 함수 ---  
REFERENCE_URL = "https://comtrade.un.org/Data/cache/partnerAreas.json"

def get_all_partner_chunks(chunk_size= 30) : 
    try : 
        # 데이터 요청 코드 
        response = requests.get(REFERENCE_URL, verify=False, timeout=60)
        response.raise_for_status() # 에러시 바로 멈추기 

        # 데이터 제이슨으로 로드해서 디코드(BOM 제거) 
        data = json.loads(response.content.decode('utf-8-sig'))

        # data 안에서 results만 끄집어내기 
        df = pd.DataFrame(data['results'])

        # id 중 숫자만 있는 것만 끄집어 내기
        df = df[df["id"].astype(str).str.isnumeric()]
        # 숫자를 리스트화 
        partner_list = df['id'].astype(str).tolist()
        # 숫자를 chunk_size로 나누기 
        partner_chunks = [partner_list[i:i + chunk_size] for i in range(0, len(partner_list), chunk_size)]

        print(f"총 {len(partner_chunks)}개의 청크로 분할 완료\n")

        return partner_chunks
    
    except Exception as e:
        print(f"Partner 코드 수집 실패 : {e}")
        return None



In [6]:
partner_chunks = get_all_partner_chunks(chunk_size=30)

총 10개의 청크로 분할 완료



**3. 데이터 수집**

In [7]:
# --- 청크에 따른 데이터 모으기 ---- 
def fetch_by_partner_chunks(year, api_key, partner_chunks) : 
    
    all_chunk_data = []

    for i, chunk in enumerate(partner_chunks) : 
        partner_str = ",".join(map(str, chunk))

        params = {
            'reporterCode' : '410', 
            'partnerCode' : partner_str, 
            'flowCode' : 'X', 
            'period' : str(year), 
            'cmdCode' : 'AG6', 
            'maxRecords' : 100000,
            'includeDesc' : 'true', 
            'subscription-key' : api_key, 
        }

        try : 
            response = requests.get(
                BASE_URL, 
                params = params, 
                verify=False, 
                timeout=120
            )
            response.raise_for_status()
            data = response.json()

            if 'data' in data and len(data['data']) > 0 : 
                df = pd.DataFrame(data['data'])
                print(f"    청크 {i+1}/{len(partner_chunks)} : {len(df):,}행 수집 완료")
                all_chunk_data.append(df)
            else : 
                print(f"    청크 {i+1}/{len(partner_chunks)} : 데이터 없음")
        
        except Exception as e : 
            print(f"    청크 {i+1}/{len(partner_chunks)} 오류 : {e}")

        time.sleep(3)

    if all_chunk_data : 
        final_df = pd.concat(all_chunk_data, ignore_index=True)
        print(f"    {year}년 전체 병합 완료 : 총 {len(final_df):,} 행")
        return final_df
    else : 
        print(f"    {year}년 : 수집된 데이터가 없습니다.")
        return None

In [8]:
# ---- 데이터 모으기 실행 ---- 
START_YEAR = 2000
END_YEAR = 2023

if not partner_chunks : 
    print('국가 코드가 없습니다.')
else : 
    print('연도별 데이터 수집 시작합니다.')

    for year in range(START_YEAR, END_YEAR + 1) : 
        filepath = f'data/raw/comtrade/kr_export_hs6_{year}.csv'

        if os.path.exists(filepath) : 
            print(f'{year} 데이터가 이미 존재함, 건너뛰기')
            continue

        print(f'{year} 수집중...')

        df = fetch_by_partner_chunks(year, COMTRADE_API_KEY, partner_chunks) 

        if df is not None : 
            df.to_csv(filepath, index=False, encoding='utf-8-sig')
            print(f'    ▶ {year}년 데이터 저장 완료')
        time.sleep(3)
print('전체 수집 완료')

연도별 데이터 수집 시작합니다.
2000 수집중...
    청크 1/10 : 12,267행 수집 완료
    청크 2/10 : 11,600행 수집 완료
    청크 3/10 : 5,128행 수집 완료
    청크 4/10 : 5,370행 수집 완료
    청크 5/10 : 15,292행 수집 완료
    청크 6/10 : 7,597행 수집 완료
    청크 7/10 : 12,441행 수집 완료
    청크 8/10 : 4,564행 수집 완료
    청크 9/10 : 11,911행 수집 완료
    청크 10/10 : 10,674행 수집 완료
    2000년 전체 병합 완료 : 총 96,844 행
    ▶ 2000년 데이터 저장 완료
2001 수집중...
    청크 1/10 : 12,219행 수집 완료
    청크 2/10 : 11,808행 수집 완료
    청크 3/10 : 5,262행 수집 완료
    청크 4/10 : 5,428행 수집 완료
    청크 5/10 : 15,912행 수집 완료
    청크 6/10 : 7,799행 수집 완료
    청크 7/10 : 12,406행 수집 완료
    청크 8/10 : 4,576행 수집 완료
    청크 9/10 : 11,885행 수집 완료
    청크 10/10 : 10,961행 수집 완료
    2001년 전체 병합 완료 : 총 98,256 행
    ▶ 2001년 데이터 저장 완료
2002 수집중...
    청크 1/10 : 12,151행 수집 완료
    청크 2/10 : 12,186행 수집 완료
    청크 3/10 : 5,460행 수집 완료
    청크 4/10 : 5,522행 수집 완료
    청크 5/10 : 16,472행 수집 완료
    청크 6/10 : 7,929행 수집 완료
    청크 7/10 : 11,289행 수집 완료
    청크 8/10 : 5,370행 수집 완료
    청크 9/10 : 12,126행 수집 완료
    청크 10/10 : 11,135행 수집 완료
    200

In [11]:
# 파일 개수 확인
csv_files = sorted([f for f in os.listdir('data/raw/comtrade') if f.endswith('.csv')])
print(f'csv파일 수 : {len(csv_files)}개')

csv파일 수 : 24개


In [ ]:
# 데이터 통합 및 확인 
df_comtrade = pd.concat(
    [pd.read_csv(f'data/raw/comtrade/{f}') for f in csv_files], 
    ignore_index = True
)

df_comtrade.head()

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20000101,2000,52,2000,410,KOR,Rep. of Korea,X,...,25.0,False,NaN,False,NaN,341.0,341.0,0,False,False
1,C,A,20000101,2000,52,2000,410,KOR,Rep. of Korea,X,...,25.0,False,NaN,False,NaN,1430.0,1430.0,0,False,False
2,C,A,20000101,2000,52,2000,410,KOR,Rep. of Korea,X,...,25.0,False,NaN,False,NaN,492.0,492.0,0,False,False
3,C,A,20000101,2000,52,2000,410,KOR,Rep. of Korea,X,...,25.0,False,NaN,False,NaN,25251.0,25251.0,0,False,False
4,C,A,20000101,2000,52,2000,410,KOR,Rep. of Korea,X,...,25.0,False,NaN,False,NaN,500.0,500.0,0,False,False


In [16]:
# 데이터 현황 확인
print(f'    전체 데이터 : {len(df_comtrade)} 행')
print(f'    기간 : {df_comtrade['period'].min()}~{df_comtrade['period'].max()}')
print(f'    상대국 수 : {df_comtrade['partnerCode'].nunique()}')
print(f'    HS품목수 : {df_comtrade['cmdCode'].nunique()}')

    전체 데이터 : 3081269 행
    기간 : 2000~2023
    상대국 수 : 239
    HS품목수 : 6349


In [25]:
# 필요한 컬럼만 추출
cols_keep = [
    'period',           # 연도
    'reporterCode',     # 보고국 코드 (410=한국)
    'reporterDesc',     # 보고국 이름
    'partnerCode',      # 상대국 코드
    'partnerDesc',      # 상대국 이름
    'cmdCode',          # HS6 코드
    'cmdDesc',          # 품목 설명
    'primaryValue',     # 수출액 (USD)
    'fobvalue',         # FOB 가격
    'netWgt',           # 순중량 (kg)
    'qty',              # 수량
]

df_clean =df_comtrade[cols_keep].copy()

# World 제거 
df_clean = df_clean[df_clean['partnerCode'] != 0]

# 컬럼 이름 변환 : cmdcode -> hs2
df_clean = df_clean.rename(columns={'cmdCode': 'hs6'})

# 대분류 컬럼 추가 
df_clean['hs2'] = df_clean['hs6'].astype(str).str[:2]


In [27]:

# 정제 데이터 저장
df_clean.to_csv('data/output/comtrade_kr_export_clean.csv', index=False, encoding='utf-8-sig')